In [1]:
import pandas as pd

df = pd.read_csv("../data/f1_features.csv")

# get the most recent row per driver from 2025
latest_features = (
    df[df["season"] == 2025]
    .sort_values(["driver", "round"])
    .groupby("driver")
    .last()
    .reset_index()
)

print(latest_features[["driver", "team", "driver_avg_finish_last3", 
                         "driver_circuit_win_rate", "team_avg_points_last3",
                         "driver_dnf_rate", "driver_championship_pos", 
                         "gap_to_leader"]].to_string())

            driver            team  driver_avg_finish_last3  driver_circuit_win_rate  team_avg_points_last3  driver_dnf_rate  driver_championship_pos  gap_to_leader
0            albon        Williams                12.666667                 0.000000               8.333333         0.179688                      8.0          309.0
1           alonso    Aston Martin                10.666667                 0.000000               2.000000         0.191638                     12.0          336.0
2        antonelli        Mercedes                 3.333333                      NaN              27.000000         0.173913                      6.0          244.0
3          bearman    Haas F1 Team                11.666667                 0.000000               3.666667         0.115385                     13.0          340.0
4        bortoleto          Sauber                16.666667                      NaN               2.666667         0.217391                     19.0          360.0
5        c

In [4]:
import numpy as np
import joblib

model = joblib.load("../models/f1_model.pkl")

# 2026 race calendar
calendar = {
    "australian": "albert_park",
    "chinese": "shanghai", 
    "japanese": "suzuka",
    "bahrain": "bahrain",
    "saudi_arabian": "jeddah",
    "miami": "miami",
    "canadian": "villeneuve",
    "monaco": "monaco",
    "barcelona": "catalunya",
    "austrian": "red_bull_ring",
    "british": "silverstone",
    "belgian": "spa",
    "hungarian": "hungaroring",
    "dutch": "zandvoort",
    "italian": "monza",
    "azerbaijan": "baku",
    "singapore": "marina_bay",
    "usgp": "americas",
    "mexico": "rodriguez",
    "brazilian": "interlagos",
    "las_vegas": "vegas",
    "qatar": "losail",
    "abu_dhabi": "yas_marina"
}

def predict_upcoming_race(circuit_id, grid_order):
    """
    Predict win probabilities for an upcoming race.
    
    circuit_id: circuit id from calendar e.g. "monaco"
    grid_order: list of driver names in qualifying order
                e.g. ["norris", "piastri", "max_verstappen"]
    """
    
    FEATURES = [
        'grid', 'driver_avg_finish_last3', 'driver_circuit_win_rate',
        'team_avg_points_last3', 'driver_dnf_rate', 'regulation_era',
        'driver_championship_pos', 'home_race', 'gap_to_leader'
    ]
    
    # build prediction dataframe
    rows = []
    for i, driver in enumerate(grid_order):
        if driver not in latest_features["driver"].values:
            print(f"Warning: {driver} not found in dataset")
            continue
            
        driver_data = latest_features[latest_features["driver"] == driver].iloc[0]
        
        # get circuit win rate for this specific circuit
        circuit_history = df[
            (df["driver"] == driver) & 
            (df["circuit"] == circuit_id)
        ]
        circuit_win_rate = circuit_history["won"].mean() if len(circuit_history) > 0 else 0.0
        
        rows.append({
            "driver": driver,
            "grid": i + 1,
            "driver_avg_finish_last3": driver_data["driver_avg_finish_last3"],
            "driver_circuit_win_rate": circuit_win_rate,
            "team_avg_points_last3": driver_data["team_avg_points_last3"],
            "driver_dnf_rate": driver_data["driver_dnf_rate"],
            "regulation_era": 3,  # 2026 is new era
            "driver_championship_pos": driver_data["driver_championship_pos"],
            "home_race": 0,
            "gap_to_leader": driver_data["gap_to_leader"]
        })
    
    pred_df = pd.DataFrame(rows)
    X = pred_df[FEATURES].fillna(0)
    
    pred_df["win_probability"] = model.predict_proba(X)[:, 1]
    pred_df["win_probability"] = pred_df["win_probability"] / pred_df["win_probability"].sum()
    
    return pred_df[["driver", "grid", "win_probability"]].sort_values(
        "win_probability", ascending=False
    )


# test — predict 2026 Australian GP
# use actual 2026 qualifying order once available
# for now use a sample grid
grid = ["norris", "max_verstappen", "piastri", "russell", "leclerc",
        "hamilton", "sainz", "alonso", "antonelli", "hadjar",
        "tsunoda", "lawson", "albon", "bearman", "hulkenberg",
        "ocon", "stroll", "gasly", "doohan", "bortoleto", "colapinto"]

results = predict_upcoming_race("albert_park", grid)
print(results.head(10))

           driver  grid  win_probability
0          norris     1         0.477567
1  max_verstappen     2         0.359283
2         piastri     3         0.094931
3         russell     4         0.046506
4         leclerc     5         0.010948
8       antonelli     9         0.005687
5        hamilton     6         0.000951
7          alonso     8         0.000839
6           sainz     7         0.000560
9          hadjar    10         0.000521


In [4]:
import pandas as pd
df = pd.read_csv(r"C:\Users\mahip\Desktop\f1-race-predictor\data\f1_features.csv")
print(sorted(df["circuit"].unique().tolist()))

['albert_park', 'americas', 'bahrain', 'baku', 'buddh', 'catalunya', 'hockenheimring', 'hungaroring', 'imola', 'interlagos', 'istanbul', 'jeddah', 'losail', 'marina_bay', 'miami', 'monaco', 'monza', 'mugello', 'nurburgring', 'portimao', 'red_bull_ring', 'ricard', 'rodriguez', 'sepang', 'shanghai', 'silverstone', 'sochi', 'spa', 'suzuka', 'valencia', 'vegas', 'villeneuve', 'yas_marina', 'yeongam', 'zandvoort']
